[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/10-engineering-toolbox/07_docker.ipynb)

*Engineering Toolbox · Notebook 7 of 8*

# Docker & Containers

Containers from the inside: image references, layer caching and instruction order, multi-stage sizes, .dockerignore, Dockerfile linting, ARG and ENV, port conflicts, compose start order and variables, restart policies, health checks, CPU and memory limits, volumes, release tags, digest pinning, compose logs, base images, and Kubernetes QoS, rolling updates and probes.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** An image is a recipe plus all its ingredients, sealed in a box; a container is the meal being cooked from it.

**What interviewers really test here:** Writing a good Dockerfile, layer caching, multi-stage builds, volumes, compose, secrets, and liveness vs readiness in Kubernetes.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Images, containers and your first Dockerfile *(Basics)*
- [ ] 2. Fast, small and safe images *(Core)*
- [ ] 3. Running a stack with Docker Compose *(Core)*
- [ ] 4. From Docker to Kubernetes *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Container vs virtual machine

|   | Container | Virtual machine |
| --- | --- | --- |
| Isolation | shares the host kernel | own kernel and OS |
| Start time | milliseconds to seconds | tens of seconds to minutes |
| Size | MBs | GBs |
| Use | packaging and running apps | strong isolation, different OSes |

### CMD vs ENTRYPOINT, COPY vs ADD

- **ENTRYPOINT** is the program that always runs; **CMD** supplies default arguments you can override at `docker run`.
- **COPY** copies local files; **ADD** can also fetch URLs and unpack archives — prefer COPY for clarity.
- Use the JSON form `CMD ["python", "app.py"]` so signals reach your process.

---
# Part 2: Lessons, step by step

## Lesson 1: Images, containers and your first Dockerfile

*Level: Basics*

"Works on my machine" happens because machines differ. A **container image** packages your code with its exact runtime and libraries; a **container** is a running instance of an image, isolated from others. The same image runs on your laptop, in CI and in production.



```python
# Dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app/ app/
RUN useradd --create-home app
USER app
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
```



```python
docker build -t invoice-api:dev .
docker run --rm -p 8000:8000 -e APP_ENV=dev invoice-api:dev
docker ps; docker logs -f <id>; docker exec -it <id> sh
```

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q3](#q3), [Q12](#q12), [Q7](#q7). Try one now, or carry on and do them in Part 3.

## Lesson 2: Fast, small and safe images

*Level: Core*

- **Layer cache:** each instruction is a cached layer; a change invalidates that layer and everything after. Copy the dependency list and install *before* copying source code, so code edits don't reinstall packages.
- **.dockerignore** keeps `.git`, `.venv`, data and secrets out of the build context.
- **Multi-stage builds** compile in a big image and copy only the results into a slim one.
- **Security:** pin base images (tag and ideally digest), run as a non-root user, clean package caches, never bake secrets into images, scan with Trivy or Docker Scout.



```python
FROM python:3.12 AS build
COPY requirements.txt .
RUN pip wheel --wheel-dir /wheels -r requirements.txt

FROM python:3.12-slim
COPY --from=build /wheels /wheels
RUN pip install --no-cache-dir /wheels/* && rm -rf /wheels
COPY app/ /app/app/
```

**✏️ Your turn:** you now have what you need for [Q9](#q9), [Q2](#q2), [Q10](#q10), [Q11](#q11), [Q5](#q5), [Q4](#q4). Try one now, or carry on and do them in Part 3.

## Lesson 3: Running a stack with Docker Compose

*Level: Core*

Compose describes several containers — API, database, cache, worker — in one file, with networks, volumes, environment and start order. Containers are disposable, so anything that must survive goes in a **volume**.



```python
# compose.yaml
services:
  db:
    image: postgres:16
    environment: {POSTGRES_PASSWORD: "${DB_PASS:?set DB_PASS}"}
    volumes: ["pgdata:/var/lib/postgresql/data"]
    healthcheck: {test: ["CMD", "pg_isready"], interval: 5s, retries: 5}
  api:
    build: .
    ports: ["8000:8000"]
    environment: {DB_URL: "postgres://postgres:${DB_PASS}@db:5432/postgres"}
    depends_on: {db: {condition: service_healthy}}
    restart: on-failure
volumes:
  pgdata:
```



```python
docker compose up -d --build; docker compose logs -f api; docker compose down
```

**✏️ Your turn:** you now have what you need for [Q13](#q13), [Q14](#q14), [Q15](#q15), [Q16](#q16), [Q17](#q17), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 4: From Docker to Kubernetes

*Level: Advanced*

Kubernetes runs containers across many machines. You declare the desired state (a **Deployment** with N replicas of an image) and it keeps it true: restarting crashed containers, spreading load, rolling out new versions gradually.




- **Requests and limits** reserve and cap CPU and memory; they also set the pod's QoS class.
- **Liveness** probes restart stuck containers; **readiness** probes stop traffic to pods that aren't ready.
- **Rolling updates** replace pods in waves controlled by `maxSurge` and `maxUnavailable`.



```python
kubectl apply -f deployment.yaml
kubectl get pods; kubectl logs deploy/api; kubectl rollout status deploy/api
kubectl rollout undo deploy/api          # instant rollback
```

**✏️ Your turn:** you now have what you need for [Q20](#q20), [Q8](#q8), [Q18](#q18), [Q19](#q19). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [2]:
import re


<a id="q1"></a>
### Q1. Read an Image Reference

🟢 Warm-up · *registry/namespace/name:tag@digest*

Split each image reference into `{registry, repository, tag, digest}` with Docker's defaults: no registry → `docker.io`; a single-name Docker Hub image gets the `library/` namespace; no tag → `latest` (unless pinned by digest, then tag is None). A first part is a registry only if it contains a `.` or `:` or is `localhost`.

**Examples**

- `refs = ['python:3.12-slim', 'nginx', 'ghcr.io/acme/api:1.4.2', 'localhost:5000/tools/cli', 'postgres@sha256:ab12cd', 'bitnami/redis:7.4']` → `[{'registry': 'docker.io', 'repository': 'library/python', 'tag': '3.12-slim', 'digest': None}, {'registry': 'docker.io', 'repository': 'library/nginx', 'tag': 'latest', 'digest': None}, {'registry': 'ghcr.io', 'repository': 'acme/api', 'tag': '1.4.2', 'digest': None}, {'registry': 'localhost:5000', 'repository': 'tools/cli', 'tag': 'latest', 'digest': None}, {'registry': 'docker.io', 'repository': 'library/postgres', 'tag': None, 'digest': 'sha256:ab12cd'}, {'registry': 'docker.io', 'repository': 'bitnami/redis', 'tag': '7.4', 'digest': None}]`

In [ ]:
def image_ref(refs):
    """
    Split each image reference into {registry, repository, tag, digest} with
    Docker's defaults: no registry → docker.io; a single-name Docker Hub image
    gets the library/ namespace; no tag → latest (unless pinned by digest, then
    tag is None). A first part is a registry only if it contains a . or : or is
    localhost.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(image_ref, [
    ((['python:3.12-slim', 'nginx', 'ghcr.io/acme/api:1.4.2', 'localhost:5000/tools/cli', 'postgres@sha256:ab12cd', 'bitnami/redis:7.4'],), [{'registry': 'docker.io', 'repository': 'library/python', 'tag': '3.12-slim', 'digest': None}, {'registry': 'docker.io', 'repository': 'library/nginx', 'tag': 'latest', 'digest': None}, {'registry': 'ghcr.io', 'repository': 'acme/api', 'tag': '1.4.2', 'digest': None}, {'registry': 'localhost:5000', 'repository': 'tools/cli', 'tag': 'latest', 'digest': None}, {'registry': 'docker.io', 'repository': 'library/postgres', 'tag': None, 'digest': 'sha256:ab12cd'}, {'registry': 'docker.io', 'repository': 'bitnami/redis', 'tag': '7.4', 'digest': None}]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Partition step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A regular expression per reference.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Partition step by step.

```python
def image_ref(refs):
    out = []
    for ref in refs:
        rest, _, digest = ref.partition("@")
        parts = rest.split("/")
        if len(parts) > 1 and ("." in parts[0] or ":" in parts[0] or parts[0] == "localhost"):
            registry, parts = parts[0], parts[1:]
        else:
            registry = "docker.io"
        path = "/".join(parts)
        tag = None
        if ":" in parts[-1]:
            path, tag = path.rsplit(":", 1)
        if registry == "docker.io" and "/" not in path:
            path = "library/" + path
        if tag is None and not digest:
            tag = "latest"
        out.append({"registry": registry, "repository": path, "tag": tag, "digest": digest or None})
    return out
```

**Cost:** O(n · L) time, O(L) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A regular expression per reference.

```python
def image_ref(refs):
    rx = re.compile(r"^(?:(?P<reg>[^/]+[.:][^/]*|localhost)/)?(?P<repo>[^:@]+)(?::(?P<tag>[^@]+))?(?:@(?P<dig>.+))?$")
    out = []
    for ref in refs:
        m = rx.match(ref)
        reg = m["reg"] or "docker.io"
        repo = m["repo"] if reg != "docker.io" or "/" in m["repo"] else "library/" + m["repo"]
        out.append({"registry": reg, "repository": repo, "tag": m["tag"] or (None if m["dig"] else "latest"), "digest": m["dig"]})
    return out
```

**Cost:** O(n · L) time, O(L) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Partition step by step. That costs O(n · L) time and O(L) space."

"I can do better: A regular expression per reference. That's O(n · L) time and O(L) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/read-an-image-reference)

<a id="q2"></a>
### Q2. Multi-Stage Builds Shrink Images

🟢 Warm-up · *Ship only what runs*

A build has stages, each a list of `(layer name, size MB)`; the final stage also lists the files it copies `from` earlier stages (`(name, size)`). A single-stage image contains every layer of every stage; a multi-stage image contains only the final stage's layers plus the copied artefacts. Return both sizes in MB (rounded to 1).

**Examples**

- `stages = {'builder': [('python:3.12 base', 1020.0), ('build-essential', 260.0), ('pip install wheels', 410.0), ('source', 3.2)], 'final': [('python:3.12-slim base', 130.0), ('app user', 0.1)]}, copied = [('site-packages', 180.0), ('source', 3.2)]` → `(1823.3, 313.3)`

In [ ]:
def image_sizes(stages, copied):
    """
    A build has stages, each a list of (layer name, size MB); the final stage
    also lists the files it copies from earlier stages ((name, size)). A single-
    stage image contains every layer of every stage; a multi-stage image
    contains only the final stage's layers plus the copied artefacts. Return
    both sizes in MB (rounded to 1).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(image_sizes, [
    (({'builder': [('python:3.12 base', 1020.0), ('build-essential', 260.0), ('pip install wheels', 410.0), ('source', 3.2)], 'final': [('python:3.12-slim base', 130.0), ('app user', 0.1)]}, [('site-packages', 180.0), ('source', 3.2)]), (1823.3, 313.3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add sizes with loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

sum() over generators.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Add sizes with loops.

```python
def image_sizes(stages, copied):
    single = 0.0
    for layers in stages.values():
        for _, size in layers:
            single += size
    multi = 0.0
    for _, size in stages["final"]:
        multi += size
    for _, size in copied:
        multi += size
    return (round(single, 1), round(multi, 1))
```

**Cost:** O(l) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** sum() over generators.

```python
def image_sizes(stages, copied):
    return (round(sum(s for layers in stages.values() for _, s in layers), 1),
            round(sum(s for _, s in stages["final"]) + sum(s for _, s in copied), 1))
```

**Cost:** O(l) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add sizes with loops. That costs O(l) time and O(1) space."

"I can do better: sum() over generators. That's O(l) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/multi-stage-builds-shrink-images)

<a id="q3"></a>
### Q3. Port Mapping Conflicts

🟢 Warm-up · *-p host:container*

Services publish ports as `'host:container'`. Two services can't bind the same host port. Return `(urls as 'service -> http://localhost:HOST' in input order, conflicts as sorted (port, [services]))`.

**Examples**

- `services = {'api': ['8000:8000'], 'web': ['3000:80'], 'admin': ['8000:8080'], 'db': ['5432:5432'], 'grafana': ['3000:3000', '9090:9090']}` → `(['api -> http://localhost:8000', 'web -> http://localhost:3000', 'admin -> http://localhost:8000', 'db -> http://localhost:5432', 'grafana -> http://localhost:3000', 'grafana -> http://localhost:9090'], [(3000, ['web', 'grafana']), (8000, ['api', 'admin'])])`

In [ ]:
from collections import defaultdict

def port_conflicts(services):
    """
    Services publish ports as 'host:container'. Two services can't bind the same
    host port. Return (urls as 'service -> http://localhost:HOST' in input
    order, conflicts as sorted (port, [services])).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(port_conflicts, [
    (({'api': ['8000:8000'], 'web': ['3000:80'], 'admin': ['8000:8080'], 'db': ['5432:5432'], 'grafana': ['3000:3000', '9090:9090']},), (['api -> http://localhost:8000', 'web -> http://localhost:3000', 'admin -> http://localhost:8000', 'db -> http://localhost:5432', 'grafana -> http://localhost:3000', 'grafana -> http://localhost:9090'], [(3000, ['web', 'grafana']), (8000, ['api', 'admin'])])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Group by host port with a dict of lists.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

defaultdict and comprehensions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Group by host port with a dict of lists.

```python
def port_conflicts(services):
    urls, used = [], {}
    for svc, ports in services.items():
        for p in ports:
            host = p.split(":")[0]
            urls.append(f"{svc} -> http://localhost:{host}")
            used.setdefault(int(host), []).append(svc)
    conflicts = []
    for port in sorted(used):
        if len(used[port]) > 1:
            conflicts.append((port, used[port]))
    return (urls, conflicts)
```

**Cost:** O(p) time, O(p) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** defaultdict and comprehensions.

```python
def port_conflicts(services):
    used = defaultdict(list)
    urls = []
    for svc, ports in services.items():
        for host in (p.split(":")[0] for p in ports):
            urls.append(f"{svc} -> http://localhost:{host}")
            used[int(host)].append(svc)
    return (urls, sorted((p, s) for p, s in used.items() if len(s) > 1))
```

**Cost:** O(p) time, O(p) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Group by host port with a dict of lists. That costs O(p) time and O(p) space."

"I can do better: defaultdict and comprehensions. That's O(p) time and O(p) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/port-mapping-conflicts)

<a id="q4"></a>
### Q4. Tag an Image for Release

🟢 Warm-up · *Traceable tags*

Given the git commit sha, branch and an optional release version, return the image tags: always `sha-` + the first 7 characters; the branch with `/` turned into `-` and lower-cased; and for a version `X.Y.Z` also `X.Y.Z`, `X.Y` and `X`, plus `latest` when the branch is `main`.

**Examples**

- `sha = '9f3a2c1d7e4b', branch = 'feature/Login-Form', version = None` → `['sha-9f3a2c1', 'feature-login-form']`
- `sha = '0b7e44aa19cc', branch = 'main', version = '1.4.2'` → `['sha-0b7e44a', 'main', '1.4.2', '1.4', '1', 'latest']`

In [ ]:
def release_tags(sha, branch, version):
    """
    Given the git commit sha, branch and an optional release version, return the
    image tags: always sha- + the first 7 characters; the branch with / turned
    into - and lower-cased; and for a version X.Y.Z also X.Y.Z, X.Y and X, plus
    latest when the branch is main.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(release_tags, [
    (('9f3a2c1d7e4b', 'feature/Login-Form', None), ['sha-9f3a2c1', 'feature-login-form']),
    (('0b7e44aa19cc', 'main', '1.4.2'), ['sha-0b7e44a', 'main', '1.4.2', '1.4', '1', 'latest']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Append tags step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Build the version tags with a slice loop.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Append tags step by step.

```python
def release_tags(sha, branch, version):
    tags = ["sha-" + sha[:7], branch.replace("/", "-").lower()]
    if version:
        major, minor, patch = version.split(".")
        tags += [version, major + "." + minor, major]
        if branch == "main":
            tags.append("latest")
    return tags
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Build the version tags with a slice loop.

```python
def release_tags(sha, branch, version):
    tags = [f"sha-{sha[:7]}", branch.replace("/", "-").lower()]
    if version:
        parts = version.split(".")
        tags += [".".join(parts[:i]) for i in (3, 2, 1)] + ["latest"] * (branch == "main")
    return tags
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Append tags step by step. That costs O(1) time and O(1) space."

"I can do better: Build the version tags with a slice loop. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/tag-an-image-for-release)

<a id="q5"></a>
### Q5. Pin Base Images by Digest

🟢 Warm-up · *Reproducible builds*

Tags like `python:3.12-slim` can be moved to new images. Rewrite each `FROM` line to pin the digest from `digests` (tag → digest), keeping any `AS name`, e.g. `FROM python:3.12-slim@sha256:… AS build`. Lines with unknown images or other instructions are unchanged.

**Examples**

- `lines = ['FROM python:3.12-slim AS build', 'RUN pip wheel -r requirements.txt', 'FROM gcr.io/distroless/python3', 'FROM python:3.12-slim', 'COPY --from=build /wheels /wheels'], digests = {'python:3.12-slim': 'sha256:4c1fe2', 'gcr.io/distroless/python3': 'sha256:9ab077'}` → `['FROM python:3.12-slim@sha256:4c1fe2 AS build', 'RUN pip wheel -r requirements.txt', 'FROM gcr.io/distroless/python3@sha256:9ab077', 'FROM python:3.12-slim@sha256:4c1fe2', 'COPY --from=build /wheels /wheels']`

In [ ]:
def pin_digests(lines, digests):
    """
    Tags like python:3.12-slim can be moved to new images. Rewrite each FROM
    line to pin the digest from digests (tag → digest), keeping any AS name,
    e.g. FROM python:3.12-slim@sha256:… AS build. Lines with unknown images or
    other instructions are unchanged.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pin_digests, [
    ((['FROM python:3.12-slim AS build', 'RUN pip wheel -r requirements.txt', 'FROM gcr.io/distroless/python3', 'FROM python:3.12-slim', 'COPY --from=build /wheels /wheels'], {'python:3.12-slim': 'sha256:4c1fe2', 'gcr.io/distroless/python3': 'sha256:9ab077'}), ['FROM python:3.12-slim@sha256:4c1fe2 AS build', 'RUN pip wheel -r requirements.txt', 'FROM gcr.io/distroless/python3@sha256:9ab077', 'FROM python:3.12-slim@sha256:4c1fe2', 'COPY --from=build /wheels /wheels']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Split each FROM line and rebuild it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One regex substitution per line.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Split each FROM line and rebuild it.

```python
def pin_digests(lines, digests):
    out = []
    for line in lines:
        parts = line.split()
        if parts[0] == "FROM" and parts[1] in digests:
            parts[1] = parts[1] + "@" + digests[parts[1]]
            out.append(" ".join(parts))
        else:
            out.append(line)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One regex substitution per line.

```python
def pin_digests(lines, digests):
    return [re.sub(r"^FROM (\S+)", lambda m: f"FROM {m[1]}@{digests[m[1]]}" if m[1] in digests else m[0], l) for l in lines]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Split each FROM line and rebuild it. That costs O(n) time and O(n) space."

"I can do better: One regex substitution per line. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/pin-base-images-by-digest)

<a id="q6"></a>
### Q6. Read docker compose logs

🟢 Warm-up · *Interleave by time*

Each container's log is a list of `(seconds, line)` in order. Merge them by time (ties: container name order) into lines formatted `'name | line'` with names padded to the longest name, like `docker compose logs`.

**Examples**

- `logs = {'api-1': [(1.0, 'started'), (3.5, 'GET /orders 200'), (4.0, 'GET /health 200')], 'worker-1': [(1.2, 'connected to redis'), (3.5, 'job 17 done')], 'db-1': [(0.4, 'ready to accept connections')]}` → `['db-1     | ready to accept connections', 'api-1    | started', 'worker-1 | connected to redis', 'api-1    | GET /orders 200', 'worker-1 | job 17 done', 'api-1    | GET /health 200']`

In [ ]:
import heapq

def compose_logs(logs):
    """
    Each container's log is a list of (seconds, line) in order. Merge them by
    time (ties: container name order) into lines formatted 'name | line' with
    names padded to the longest name, like docker compose logs.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(compose_logs, [
    (({'api-1': [(1.0, 'started'), (3.5, 'GET /orders 200'), (4.0, 'GET /health 200')], 'worker-1': [(1.2, 'connected to redis'), (3.5, 'job 17 done')], 'db-1': [(0.4, 'ready to accept connections')]},), ['db-1     | ready to accept connections', 'api-1    | started', 'worker-1 | connected to redis', 'api-1    | GET /orders 200', 'worker-1 | job 17 done', 'api-1    | GET /health 200']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Collect everything and sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

heapq.merge of the already-sorted streams.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Collect everything and sort.

```python
def compose_logs(logs):
    width = max(len(n) for n in logs)
    rows = []
    for name, lines in logs.items():
        for t, line in lines:
            rows.append((t, name, line))
    rows.sort()
    return [name.ljust(width) + " | " + line for _, name, line in rows]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** heapq.merge of the already-sorted streams.

```python
def compose_logs(logs):
    width = max(map(len, logs))
    streams = [[(t, n, l) for t, l in lines] for n, lines in logs.items()]
    return [f"{n:<{width}} | {l}" for _, n, l in heapq.merge(*streams)]
```

**Cost:** O(n log c) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Collect everything and sort. That costs O(n log n) time and O(n) space."

"I can do better: heapq.merge of the already-sorted streams. That's O(n log c) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/read-docker-compose-logs)

<a id="q7"></a>
### Q7. Pick a Base Image

🟢 Warm-up · *Smallest that works*

Each base in `BASES` is `(name, MB, features)`. For each app's needed features, return the smallest base providing all of them. (Alpine uses musl instead of glibc — many Python wheels need glibc.)

**Examples**

- `needs = [[], ['glibc'], ['shell'], ['glibc', 'shell'], ['glibc', 'tools']]` → `['scratch', 'distroless', 'alpine', 'debian-slim', 'ubuntu']`

In [ ]:
BASES = [("scratch", 0, set()), ("distroless", 25, {"glibc"}), ("alpine", 8, {"shell", "pkg"}),
         ("debian-slim", 75, {"glibc", "shell", "pkg"}), ("ubuntu", 78, {"glibc", "shell", "pkg", "tools"})]

def pick_base(needs):
    """
    Each base in BASES is (name, MB, features). For each app's needed features,
    return the smallest base providing all of them. (Alpine uses musl instead of
    glibc — many Python wheels need glibc.)
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pick_base, [
    (([[], ['glibc'], ['shell'], ['glibc', 'shell'], ['glibc', 'tools']],), ['scratch', 'distroless', 'alpine', 'debian-slim', 'ubuntu']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan bases, keep the smallest match.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

min() with a key over matching bases.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Scan bases, keep the smallest match.

```python
def pick_base(needs):
    out = []
    for need in needs:
        best = None
        for name, size, feats in BASES:
            if set(need) <= feats and (best is None or size < best[1]):
                best = (name, size)
        out.append(best[0])
    return out
```

**Cost:** O(a · b) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** min() with a key over matching bases.

```python
def pick_base(needs):
    return [min((b for b in BASES if set(n) <= b[2]), key=lambda b: b[1])[0] for n in needs]
```

**Cost:** O(a · b) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan bases, keep the smallest match. That costs O(a · b) time and O(1) space."

"I can do better: min() with a key over matching bases. That's O(a · b) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/pick-a-base-image)

<a id="q8"></a>
### Q8. Kubernetes QoS Classes

🟢 Warm-up · *Who gets evicted first?*

A pod's QoS class comes from its containers' CPU and memory `requests` and `limits`: `Guaranteed` if every container sets both limits and requests equal to them; `BestEffort` if no container sets any; otherwise `Burstable`. Under memory pressure BestEffort pods are evicted first.

**Examples**

- `pods = [[{'requests': {'cpu': '500m', 'memory': '512Mi'}, 'limits': {'cpu': '500m', 'memory': '512Mi'}}], [{'requests': {'cpu': '250m'}, 'limits': {'memory': '1Gi'}}], [{'requests': {}, 'limits': {}}, {'requests': {}, 'limits': {}}], [{'requests': {'cpu': '1', 'memory': '1Gi'}, 'limits': {'cpu': '1', 'memory': '1Gi'}}, {'requests': {}, 'limits': {}}]]` → `['Guaranteed', 'Burstable', 'BestEffort', 'Burstable']`

In [ ]:
def qos(pods):
    """
    A pod's QoS class comes from its containers' CPU and memory requests and
    limits: Guaranteed if every container sets both limits and requests equal to
    them; BestEffort if no container sets any; otherwise Burstable. Under memory
    pressure BestEffort pods are evicted first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(qos, [
    (([[{'requests': {'cpu': '500m', 'memory': '512Mi'}, 'limits': {'cpu': '500m', 'memory': '512Mi'}}], [{'requests': {'cpu': '250m'}, 'limits': {'memory': '1Gi'}}], [{'requests': {}, 'limits': {}}, {'requests': {}, 'limits': {}}], [{'requests': {'cpu': '1', 'memory': '1Gi'}, 'limits': {'cpu': '1', 'memory': '1Gi'}}, {'requests': {}, 'limits': {}}]],), ['Guaranteed', 'Burstable', 'BestEffort', 'Burstable']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Flags per container.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

all()/any() over containers.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Flags per container.

```python
def qos(pods):
    out = []
    for containers in pods:
        any_set = False
        guaranteed = True
        for c in containers:
            if c["requests"] or c["limits"]:
                any_set = True
            for r in ("cpu", "memory"):
                if r not in c["limits"] or c["requests"].get(r, c["limits"][r]) != c["limits"][r]:
                    guaranteed = False
        out.append("Guaranteed" if guaranteed else "Burstable" if any_set else "BestEffort")
    return out
```

**Cost:** O(p · c) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** all()/any() over containers.

```python
def qos(pods):
    def guaranteed(c):
        return all(r in c["limits"] and c["requests"].get(r, c["limits"][r]) == c["limits"][r] for r in ("cpu", "memory"))
    return ["Guaranteed" if all(map(guaranteed, p)) else "Burstable" if any(c["requests"] or c["limits"] for c in p) else "BestEffort" for p in pods]
```

**Cost:** O(p · c) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Flags per container. That costs O(p · c) time and O(1) space."

"I can do better: all()/any() over containers. That's O(p · c) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `pods = [[{'requests': {}, 'limits': {}}]]`

1. GeneratorExit raised: 
2. guaranteed() returns False
3. qos() returns ['BestEffort']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/kubernetes-qos-classes)

<a id="q9"></a>
### Q9. Which Layers Rebuild?

🟡 Interview standard · *Order your Dockerfile for the cache*

Each Dockerfile step is `(instruction, files it copies)`. Docker reuses cached layers until the first step whose inputs changed (a COPY of a changed file); that step and **every later step** rebuild. Return the indexes (0-based) of rebuilt steps for each set of changed files — compare copying everything early with copying the dependency list first.

**Examples**

- `steps = [('FROM python:3.12-slim', []), ('WORKDIR /app', []), ('COPY . .', ['requirements.txt', 'app/main.py', 'app/db.py']), ('RUN pip install -r requirements.txt', []), ('CMD uvicorn app.main:app', [])], changes = [['app/main.py'], ['requirements.txt']]` → `[[2, 3, 4], [2, 3, 4]]`
- `steps = [('FROM python:3.12-slim', []), ('WORKDIR /app', []), ('COPY requirements.txt .', ['requirements.txt']), ('RUN pip install -r requirements.txt', []), ('COPY app/ app/', ['app/main.py', 'app/db.py']), ('CMD uvicorn app.main:app', [])], changes = [['app/main.py'], ['requirements.txt']]` → `[[4, 5], [2, 3, 4, 5]]`

In [ ]:
def rebuilt_steps(steps, changes):
    """
    Each Dockerfile step is (instruction, files it copies). Docker reuses cached
    layers until the first step whose inputs changed (a COPY of a changed file);
    that step and every later step rebuild. Return the indexes (0-based) of
    rebuilt steps for each set of changed files — compare copying everything
    early with copying the dependency list first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rebuilt_steps, [
    (([('FROM python:3.12-slim', []), ('WORKDIR /app', []), ('COPY . .', ['requirements.txt', 'app/main.py', 'app/db.py']), ('RUN pip install -r requirements.txt', []), ('CMD uvicorn app.main:app', [])], [['app/main.py'], ['requirements.txt']]), [[2, 3, 4], [2, 3, 4]]),
    (([('FROM python:3.12-slim', []), ('WORKDIR /app', []), ('COPY requirements.txt .', ['requirements.txt']), ('RUN pip install -r requirements.txt', []), ('COPY app/ app/', ['app/main.py', 'app/db.py']), ('CMD uvicorn app.main:app', [])], [['app/main.py'], ['requirements.txt']]), [[4, 5], [2, 3, 4, 5]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan steps for each change set.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

next() for the first invalidated step, then a range.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Scan steps for each change set.

```python
def rebuilt_steps(steps, changes):
    out = []
    for changed in changes:
        first = None
        for i, (_, files) in enumerate(steps):
            for f in files:
                if f in changed and first is None:
                    first = i
        out.append([] if first is None else list(range(first, len(steps))))
    return out
```

**Cost:** O(c · s · f) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** next() for the first invalidated step, then a range.

```python
def rebuilt_steps(steps, changes):
    out = []
    for changed in changes:
        first = next((i for i, (_, files) in enumerate(steps) if set(files) & set(changed)), len(steps))
        out.append(list(range(first, len(steps))))
    return out
```

**Cost:** O(c · s · f) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan steps for each change set. That costs O(c · s · f) time and O(s) space."

"I can do better: next() for the first invalidated step, then a range. That's O(c · s · f) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/which-layers-rebuild)

<a id="q10"></a>
### Q10. What Goes Into the Build Context?

🟡 Interview standard · *.dockerignore*

Docker sends the build context (the folder) to the builder, minus `.dockerignore` matches. Patterns match from the context root (`*` doesn't cross `/`, `**` matches any number of folders), a pattern also excludes everything inside a matching folder, and `!` re-includes; the last match wins. Return `(files sent, MB sent rounded to 1, MB saved rounded to 1)`.

**Examples**

- `files = {'.git/objects/pack/p1': 380.0, '.venv/lib/torch/lib.so': 2100.0, 'app/main.py': 0.02, 'app/db.py': 0.01, 'tests/test_api.py': 0.01, 'data/train.parquet': 850.0, 'data/README.md': 0.01, 'requirements.txt': 0.001, 'notes/todo.log': 0.3, 'app/debug.log': 0.2}, patterns = ['.git', '.venv', 'data', '!data/README.md', '**/*.log', 'tests']` → `(['app/db.py', 'app/main.py', 'data/README.md', 'requirements.txt'], 0.0, 3330.5)`

In [ ]:
def build_context(files, patterns):
    """
    Docker sends the build context (the folder) to the builder, minus
    .dockerignore matches. Patterns match from the context root (* doesn't cross
    /, ** matches any number of folders), a pattern also excludes everything
    inside a matching folder, and ! re-includes; the last match wins. Return
    (files sent, MB sent rounded to 1, MB saved rounded to 1).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(build_context, [
    (({'.git/objects/pack/p1': 380.0, '.venv/lib/torch/lib.so': 2100.0, 'app/main.py': 0.02, 'app/db.py': 0.01, 'tests/test_api.py': 0.01, 'data/train.parquet': 850.0, 'data/README.md': 0.01, 'requirements.txt': 0.001, 'notes/todo.log': 0.3, 'app/debug.log': 0.2}, ['.git', '.venv', 'data', '!data/README.md', '**/*.log', 'tests']), (['app/db.py', 'app/main.py', 'data/README.md', 'requirements.txt'], 0.0, 3330.5)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Match with fnmatch on every prefix of the path.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A compiled regex per pattern.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Match with fnmatch on every prefix of the path.

```python
def ignore_rx(pattern):
    r = ""
    for tok in re.split(r"(\*\*/|\*\*|\*|\?)", pattern.strip("/")):
        r += {"**/": "(?:.*/)?", "**": ".*", "*": "[^/]*", "?": "[^/]"}.get(tok, re.escape(tok))
    return re.compile(r + "(?:/.*)?$")


def build_context(files, patterns):
    sent = []
    for path in sorted(files):
        excluded = False
        for p in patterns:
            neg = p.startswith("!")
            if ignore_rx(p.lstrip("!")).match(path):
                excluded = not neg
        if not excluded:
            sent.append(path)
    total = sum(files.values())
    size = sum(files[p] for p in sent)
    return (sent, round(size, 1), round(total - size, 1))
```

**Cost:** O(f · p · d) time, O(f) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A compiled regex per pattern.

```python
def ignore_rx(pattern):
    r = ""
    for tok in re.split(r"(\*\*/|\*\*|\*|\?)", pattern.strip("/")):
        r += {"**/": "(?:.*/)?", "**": ".*", "*": "[^/]*", "?": "[^/]"}.get(tok, re.escape(tok))
    return re.compile(r + "(?:/.*)?$")


def build_context(files, patterns):
    rules = [(p.startswith("!"), ignore_rx(p.lstrip("!"))) for p in reversed(patterns)]
    keep = lambda path: next((neg for neg, rx in rules if rx.match(path)), True)
    sent = sorted(p for p in files if keep(p))
    size = sum(files[p] for p in sent)
    return (sent, round(size, 1), round(sum(files.values()) - size, 1))
```

**Cost:** O(f · p) time, O(f) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Match with fnmatch on every prefix of the path. That costs O(f · p · d) time and O(f) space."

"I can do better: A compiled regex per pattern. That's O(f · p) time and O(f) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/what-goes-into-the-build-context)

<a id="q11"></a>
### Q11. Lint a Dockerfile

🟡 Interview standard · *hadolint-style rules*

Return sorted `(line, rule)` findings: `'pin base image'` (FROM without a tag, or with `:latest`), `'clean apt lists'` (RUN with `apt-get install` but no `rm -rf /var/lib/apt/lists`), `'pip no-cache'` (pip install without `--no-cache-dir`), `'use COPY'` (ADD of a local path, not a URL), and `'runs as root'` (no USER instruction anywhere; report line 0).

**Examples**

- `lines = ['FROM python:latest', 'RUN apt-get update && apt-get install -y gcc', 'ADD . /app', 'RUN pip install -r /app/requirements.txt', 'ADD https://example.com/model.bin /models/', 'CMD ["python", "/app/main.py"]']` → `[(0, 'runs as root'), (1, 'pin base image'), (2, 'clean apt lists'), (3, 'use COPY'), (4, 'pip no-cache')]`
- `lines = ['FROM python:3.12-slim', 'RUN apt-get update && apt-get install -y --no-install-recommends gcc && rm -rf /var/lib/apt/lists/*', 'COPY . /app', 'RUN pip install --no-cache-dir -r /app/requirements.txt', 'USER app', 'CMD ["python", "/app/main.py"]']` → `[]`

In [ ]:
def lint_dockerfile(lines):
    """
    Return sorted (line, rule) findings: 'pin base image' (FROM without a tag,
    or with :latest), 'clean apt lists' (RUN with apt-get install but no rm -rf
    /var/lib/apt/lists), 'pip no-cache' (pip install without --no-cache-dir),
    'use COPY' (ADD of a local path, not a URL), and 'runs as root' (no USER
    instruction anywhere; report line 0).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lint_dockerfile, [
    ((['FROM python:latest', 'RUN apt-get update && apt-get install -y gcc', 'ADD . /app', 'RUN pip install -r /app/requirements.txt', 'ADD https://example.com/model.bin /models/', 'CMD ["python", "/app/main.py"]'],), [(0, 'runs as root'), (1, 'pin base image'), (2, 'clean apt lists'), (3, 'use COPY'), (4, 'pip no-cache')]),
    ((['FROM python:3.12-slim', 'RUN apt-get update && apt-get install -y --no-install-recommends gcc && rm -rf /var/lib/apt/lists/*', 'COPY . /app', 'RUN pip install --no-cache-dir -r /app/requirements.txt', 'USER app', 'CMD ["python", "/app/main.py"]'],), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

If-statements per instruction.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A table of (instruction, check, rule).

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** If-statements per instruction.

```python
def lint_dockerfile(lines):
    out, has_user = [], False
    for i, line in enumerate(lines, 1):
        ins, _, rest = line.partition(" ")
        if ins == "FROM":
            image = rest.split()[0]
            if ":" not in image.split("/")[-1] or image.endswith(":latest"):
                out.append((i, "pin base image"))
        if ins == "RUN" and "apt-get install" in rest and "rm -rf /var/lib/apt/lists" not in rest:
            out.append((i, "clean apt lists"))
        if ins == "RUN" and "pip install" in rest and "--no-cache-dir" not in rest:
            out.append((i, "pip no-cache"))
        if ins == "ADD" and not rest.startswith(("http://", "https://")):
            out.append((i, "use COPY"))
        if ins == "USER":
            has_user = True
    if not has_user:
        out.append((0, "runs as root"))
    return sorted(out)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A table of (instruction, check, rule).

```python
def lint_dockerfile(lines):
    rules = [("FROM", lambda r: ":" not in r.split()[0].split("/")[-1] or r.split()[0].endswith(":latest"), "pin base image"),
             ("RUN", lambda r: "apt-get install" in r and "rm -rf /var/lib/apt/lists" not in r, "clean apt lists"),
             ("RUN", lambda r: "pip install" in r and "--no-cache-dir" not in r, "pip no-cache"),
             ("ADD", lambda r: not r.startswith(("http://", "https://")), "use COPY")]
    parsed = [l.partition(" ")[::2] for l in lines]
    out = [(i, name) for i, (ins, rest) in enumerate(parsed, 1) for kind, bad, name in rules if ins == kind and bad(rest)]
    if all(ins != "USER" for ins, _ in parsed):
        out.append((0, "runs as root"))
    return sorted(out)
```

**Cost:** O(n · r) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: If-statements per instruction. That costs O(n) time and O(n) space."

"I can do better: A table of (instruction, check, rule). That's O(n · r) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/lint-a-dockerfile)

<a id="q12"></a>
### Q12. Resolve ARG and ENV

🟡 Interview standard · *Build-time vs run-time variables*

Process `ARG name=default` and `ENV name=value` lines in order. `${name}` in a value expands from the ENV and ARG values seen so far (ENV wins). `build_args` (from `--build-arg`) override ARG defaults. Return `(environment variables in the final image, sorted items)` — ARGs don't survive into the running container unless copied into an ENV.

**Examples**

- `lines = ['ARG PYTHON_VERSION=3.12', 'ARG APP_VERSION=dev', 'ENV APP_HOME=/srv/app', 'ENV PATH=${APP_HOME}/bin:/usr/bin', 'ENV APP_VERSION=${APP_VERSION}', 'ARG SECRET_TOKEN', 'ENV PYTHONUNBUFFERED=1'], build_args = {'APP_VERSION': '1.4.2', 'SECRET_TOKEN': 'abc'}` → `[('APP_HOME', '/srv/app'), ('APP_VERSION', '1.4.2'), ('PATH', '/srv/app/bin:/usr/bin'), ('PYTHONUNBUFFERED', '1')]`

In [ ]:
def docker_env(lines, build_args):
    """
    Process ARG name=default and ENV name=value lines in order. ${name} in a
    value expands from the ENV and ARG values seen so far (ENV wins). build_args
    (from --build-arg) override ARG defaults. Return (environment variables in
    the final image, sorted items) — ARGs don't survive into the running
    container unless copied into an ENV.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(docker_env, [
    ((['ARG PYTHON_VERSION=3.12', 'ARG APP_VERSION=dev', 'ENV APP_HOME=/srv/app', 'ENV PATH=${APP_HOME}/bin:/usr/bin', 'ENV APP_VERSION=${APP_VERSION}', 'ARG SECRET_TOKEN', 'ENV PYTHONUNBUFFERED=1'], {'APP_VERSION': '1.4.2', 'SECRET_TOKEN': 'abc'}), [('APP_HOME', '/srv/app'), ('APP_VERSION', '1.4.2'), ('PATH', '/srv/app/bin:/usr/bin'), ('PYTHONUNBUFFERED', '1')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two dicts and a manual \${} scanner.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

re.sub with a lookup function.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Two dicts and a manual \${} scanner.

```python
def docker_env(lines, build_args):
    args, env = {}, {}
    for line in lines:
        ins, _, rest = line.partition(" ")
        name, _, value = rest.partition("=")
        while "${" in value:
            s = value.index("${")
            e = value.index("}", s)
            key = value[s + 2:e]
            value = value[:s] + env.get(key, args.get(key, "")) + value[e + 1:]
        if ins == "ARG":
            args[name] = build_args.get(name, value)
        else:
            env[name] = value
    return sorted(env.items())
```

**Cost:** O(n · L) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** re.sub with a lookup function.

```python
def docker_env(lines, build_args):
    args, env = {}, {}
    for line in lines:
        ins, _, rest = line.partition(" ")
        name, _, value = rest.partition("=")
        value = re.sub(r"\$\{(\w+)\}", lambda m: env.get(m[1], args.get(m[1], "")), value)
        if ins == "ARG":
            args[name] = build_args.get(name, value)
        else:
            env[name] = value
    return sorted(env.items())
```

**Cost:** O(n · L) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two dicts and a manual \${} scanner. That costs O(n · L) time and O(n) space."

"I can do better: re.sub with a lookup function. That's O(n · L) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/resolve-arg-and-env)

<a id="q13"></a>
### Q13. Compose Start Order

🟡 Interview standard · *depends\_on with health checks*

Each service has `depends_on` (services that must be **healthy** first) and seconds it takes from start to healthy. A service starts as soon as all its dependencies are healthy. Return `{service: [start second, healthy second]}` and the time until everything is healthy.

**Examples**

- `deps = {'db': [], 'redis': [], 'migrate': ['db'], 'api': ['db', 'redis', 'migrate'], 'worker': ['redis', 'db'], 'web': ['api']}, ready = {'db': 8, 'redis': 2, 'migrate': 5, 'api': 4, 'worker': 3, 'web': 1}` → `({'db': [0, 8], 'redis': [0, 2], 'migrate': [8, 13], 'api': [13, 17], 'worker': [8, 11], 'web': [17, 18]}, 18)`

In [ ]:
def compose_order(deps, ready):
    """
    Each service has depends_on (services that must be healthy first) and
    seconds it takes from start to healthy. A service starts as soon as all its
    dependencies are healthy. Return {service: [start second, healthy second]}
    and the time until everything is healthy.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(compose_order, [
    (({'db': [], 'redis': [], 'migrate': ['db'], 'api': ['db', 'redis', 'migrate'], 'worker': ['redis', 'db'], 'web': ['api']}, {'db': 8, 'redis': 2, 'migrate': 5, 'api': 4, 'worker': 3, 'web': 1}), ({'db': [0, 8], 'redis': [0, 2], 'migrate': [8, 13], 'api': [13, 17], 'worker': [8, 11], 'web': [17, 18]}, 18)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeatedly compute start times until nothing changes.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Memoised recursion: start = max of dependencies' healthy times.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Repeatedly compute start times until nothing changes.

```python
def compose_order(deps, ready):
    start = {s: 0 for s in deps}
    changed = True
    while changed:
        changed = False
        for s, ds in deps.items():
            t = max([start[d] + ready[d] for d in ds], default=0)
            if t != start[s]:
                start[s], changed = t, True
    out = {s: [start[s], start[s] + ready[s]] for s in deps}
    return (out, max(v[1] for v in out.values()))
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Memoised recursion: start = max of dependencies' healthy times.

```python
def compose_order(deps, ready):
    from functools import lru_cache
    @lru_cache(None)
    def healthy(s):
        return max((healthy(d) for d in deps[s]), default=0) + ready[s]
    out = {s: [healthy(s) - ready[s], healthy(s)] for s in deps}
    return (out, max(healthy(s) for s in deps))
```

**Cost:** O(n + e) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeatedly compute start times until nothing changes. That costs O(n²) time and O(n) space."

"I can do better: Memoised recursion: start = max of dependencies' healthy times. That's O(n + e) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `deps = {'a': [], 'b': ['a']}, ready = {'a': 2, 'b': 1}`

1. healthy() returns 2
2. healthy() returns 3
3. compose\_order() returns ({'a': [0, 2], 'b': [2, 3]}, 3)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/compose-start-order)

<a id="q14"></a>
### Q14. Interpolate Compose Variables

🟡 Interview standard · *\${VAR:-default} and \${VAR:?error}*

Docker Compose fills `${VAR}` from the environment, `${VAR:-default}` uses the default when unset or empty, and `${VAR:?message}` stops with an error when unset or empty. Return each resolved value, or `'error: message'`.

**Examples**

- `values = ['postgres://${DB_USER}:${DB_PASS:?set DB_PASS}@db:5432/${DB_NAME:-app}', '${API_PORT:-8000}', '${IMAGE_TAG:?IMAGE_TAG is required}', 'redis://${REDIS_HOST:-redis}:6379'], env = {'DB_USER': 'app', 'DB_PASS': 's3cret', 'REDIS_HOST': '', 'IMAGE_TAG': ''}` → `['postgres://app:s3cret@db:5432/app', '8000', 'error: IMAGE_TAG is required', 'redis://redis:6379']`

In [ ]:
def compose_vars(values, env):
    """
    Docker Compose fills ${VAR} from the environment, ${VAR:-default} uses the
    default when unset or empty, and ${VAR:?message} stops with an error when
    unset or empty. Return each resolved value, or 'error: message'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(compose_vars, [
    ((['postgres://${DB_USER}:${DB_PASS:?set DB_PASS}@db:5432/${DB_NAME:-app}', '${API_PORT:-8000}', '${IMAGE_TAG:?IMAGE_TAG is required}', 'redis://${REDIS_HOST:-redis}:6379'], {'DB_USER': 'app', 'DB_PASS': 's3cret', 'REDIS_HOST': '', 'IMAGE_TAG': ''}), ['postgres://app:s3cret@db:5432/app', '8000', 'error: IMAGE_TAG is required', 'redis://redis:6379']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan for \${...} with index().

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

re.sub with a function that may raise.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Scan for \${...} with index().

```python
def compose_vars(values, env):
    out = []
    for v in values:
        result, error = "", None
        i = 0
        while i < len(v):
            if v.startswith("${", i):
                j = v.index("}", i)
                inner = v[i + 2:j]
                if ":?" in inner:
                    name, msg = inner.split(":?", 1)
                    if not env.get(name):
                        error = msg
                        break
                    result += env[name]
                elif ":-" in inner:
                    name, default = inner.split(":-", 1)
                    result += env.get(name) or default
                else:
                    result += env.get(inner, "")
                i = j + 1
            else:
                result += v[i]
                i += 1
        out.append(f"error: {error}" if error else result)
    return out
```

**Cost:** O(n · L) time, O(L) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** re.sub with a function that may raise.

```python
def compose_vars(values, env):
    def sub(m):
        name, op, arg = m.group(1), m.group(2), m.group(3)
        val = env.get(name, "")
        if op == ":?" and not val:
            raise ValueError(arg)
        return val or (arg if op == ":-" else "")
    out = []
    for v in values:
        try:
            out.append(re.sub(r"\$\{(\w+)(?:(:-|:\?)([^}]*))?\}", sub, v))
        except ValueError as e:
            out.append(f"error: {e}")
    return out
```

**Cost:** O(n · L) time, O(L) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan for \${...} with index(). That costs O(n · L) time and O(L) space."

"I can do better: re.sub with a function that may raise. That's O(n · L) time and O(L) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/interpolate-compose-variables)

<a id="q15"></a>
### Q15. Restart Policies

🟡 Interview standard · *no, on-failure, always*

A container exits with each code in `exits` in turn (it would run again if restarted). Policies: `'no'` never restarts; `'on-failure:N'` restarts after non-zero exits, at most N times; `'always'` restarts after any exit. Return `(restarts, final exit code)` for each policy. (Docker also adds growing delays between restarts.)

**Examples**

- `exits = [1, 1, 137, 0, 2], policies = ['no', 'on-failure:2', 'on-failure:5', 'always']` → `[(0, 1), (2, 137), (3, 0), (4, 2)]`

In [ ]:
def restarts(exits, policies):
    """
    A container exits with each code in exits in turn (it would run again if
    restarted). Policies: 'no' never restarts; 'on-failure:N' restarts after
    non-zero exits, at most N times; 'always' restarts after any exit. Return
    (restarts, final exit code) for each policy. (Docker also adds growing
    delays between restarts.)
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(restarts, [
    (([1, 1, 137, 0, 2], ['no', 'on-failure:2', 'on-failure:5', 'always']), [(0, 1), (2, 137), (3, 0), (4, 2)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate each policy with a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Parse the policy into (restart test, limit) first.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Simulate each policy with a loop.

```python
def restarts(exits, policies):
    out = []
    for pol in policies:
        n, i = 0, 0
        while True:
            code = exits[i]
            if pol == "no":
                break
            if pol == "always":
                ok = True
            else:
                ok = code != 0 and n < int(pol.split(":")[1])
            if not ok or i == len(exits) - 1:
                break
            n += 1
            i += 1
        out.append((n, exits[i]))
    return out
```

**Cost:** O(p · e) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Parse the policy into (restart test, limit) first.

```python
def restarts(exits, policies):
    out = []
    for pol in policies:
        name, _, lim = pol.partition(":")
        limit = int(lim) if lim else math.inf
        n = 0
        for i, code in enumerate(exits):
            again = name == "always" or (name == "on-failure" and code != 0 and n < limit)
            if not again or i == len(exits) - 1:
                out.append((n, code))
                break
            n += 1
    return out
```

**Cost:** O(p · e) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate each policy with a loop. That costs O(p · e) time and O(1) space."

"I can do better: Parse the policy into (restart test, limit) first. That's O(p · e) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/restart-policies)

<a id="q16"></a>
### Q16. When Is a Container Unhealthy?

🟡 Interview standard · *HEALTHCHECK settings*

A health probe runs every `interval` seconds starting at second `interval`, with results in `results`. Failures during the first `start_period` seconds don't count. The container becomes `unhealthy` after `retries` consecutive counted failures, and healthy again on the next success. Return the list of `(second, new status)` changes, starting from `'starting'` (the first success makes it `'healthy'`).

**Examples**

- `results = [False, False, False, True, True, False, False, False, True, False], interval = 10, start_period = 25, retries = 3` → `[(40, 'healthy'), (80, 'unhealthy'), (90, 'healthy')]`

In [ ]:
def health_status(results, interval, start_period, retries):
    """
    A health probe runs every interval seconds starting at second interval, with
    results in results. Failures during the first start_period seconds don't
    count. The container becomes unhealthy after retries consecutive counted
    failures, and healthy again on the next success. Return the list of (second,
    new status) changes, starting from 'starting' (the first success makes it
    'healthy').
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(health_status, [
    (([False, False, False, True, True, False, False, False, True, False], 10, 25, 3), [(40, 'healthy'), (80, 'unhealthy'), (90, 'healthy')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate probe by probe.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same with a status variable and a change list.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Simulate probe by probe.

```python
def health_status(results, interval, start_period, retries):
    status, fails, changes = "starting", 0, []
    for i, ok in enumerate(results, 1):
        t = i * interval
        if ok:
            fails = 0
            if status != "healthy":
                status = "healthy"
                changes.append((t, status))
        elif t > start_period:
            fails += 1
            if fails >= retries and status != "unhealthy":
                status = "unhealthy"
                changes.append((t, status))
    return changes
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same with a status variable and a change list.

```python
def health_status(results, interval, start_period, retries):
    status, fails, changes = "starting", 0, []
    for t, ok in zip(range(interval, interval * (len(results) + 1), interval), results):
        fails = 0 if ok else fails + (t > start_period)
        new = "healthy" if ok else "unhealthy" if fails >= retries else status
        if new != status:
            status = new
            changes.append((t, status))
    return changes
```

**Cost:** O(n) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate probe by probe. That costs O(n) time and O(1) space."

"I can do better: Same with a status variable and a change list. That's O(n) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/when-is-a-container-unhealthy)

<a id="q17"></a>
### Q17. Where Does the Data Go?

🟡 Interview standard · *Volumes, bind mounts and the container layer*

Mounts map container paths to a named volume (`'vol:NAME'`) or a host folder (`'host:/path'`). A file written in the container lands in the mount with the **longest** matching path prefix, or in the container's writable layer (`'container layer (lost on removal)'`). Return where each written path ends up.

**Examples**

- `mounts = {'/var/lib/postgresql/data': 'vol:pgdata', '/app': 'host:/home/asha/proj', '/app/uploads': 'vol:uploads'}, writes = ['/var/lib/postgresql/data/base/1', '/app/main.py', '/app/uploads/a.png', '/tmp/cache.bin', '/application/x']` → `{'/var/lib/postgresql/data/base/1': 'volume pgdata', '/app/main.py': '/home/asha/proj/main.py', '/app/uploads/a.png': 'volume uploads', '/tmp/cache.bin': 'container layer (lost on removal)', '/application/x': 'container layer (lost on removal)'}`

In [ ]:
def data_location(mounts, writes):
    """
    Mounts map container paths to a named volume ('vol:NAME') or a host folder
    ('host:/path'). A file written in the container lands in the mount with the
    longest matching path prefix, or in the container's writable layer
    ('container layer (lost on removal)'). Return where each written path ends
    up.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(data_location, [
    (({'/var/lib/postgresql/data': 'vol:pgdata', '/app': 'host:/home/asha/proj', '/app/uploads': 'vol:uploads'}, ['/var/lib/postgresql/data/base/1', '/app/main.py', '/app/uploads/a.png', '/tmp/cache.bin', '/application/x']), {'/var/lib/postgresql/data/base/1': 'volume pgdata', '/app/main.py': '/home/asha/proj/main.py', '/app/uploads/a.png': 'volume uploads', '/tmp/cache.bin': 'container layer (lost on removal)', '/application/x': 'container layer (lost on removal)'}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check every mount with startswith.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort mounts by length once, take the first match.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Check every mount with startswith.

```python
def data_location(mounts, writes):
    out = {}
    for f in writes:
        best = None
        for path, target in mounts.items():
            if (f == path or f.startswith(path + "/")) and (best is None or len(path) > len(best)):
                best = path
        if best is None:
            out[f] = "container layer (lost on removal)"
        else:
            kind, name = mounts[best].split(":", 1)
            out[f] = f"volume {name}" if kind == "vol" else name + f[len(best):]
    return out
```

**Cost:** O(f · m) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Sort mounts by length once, take the first match.

```python
def data_location(mounts, writes):
    ordered = sorted(mounts, key=len, reverse=True)
    def where(f):
        p = next((p for p in ordered if f == p or f.startswith(p + "/")), None)
        if p is None:
            return "container layer (lost on removal)"
        kind, name = mounts[p].split(":", 1)
        return f"volume {name}" if kind == "vol" else name + f[len(p):]
    return {f: where(f) for f in writes}
```

**Cost:** O(m log m + f · m) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check every mount with startswith. That costs O(f · m) time and O(1) space."

"I can do better: Sort mounts by length once, take the first match. That's O(m log m + f · m) time and O(m) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `mounts = {'/data': 'vol:d'}, writes = ['/data/x', '/tmp/y']`

1. GeneratorExit raised: 
2. where() returns 'volume d'
3. where() returns 'container layer (lost on removal)'
4. data\_location() returns {'/data/x': 'volume d', '/tmp/y': 'co...

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/where-does-the-data-go)

<a id="q18"></a>
### Q18. Plan a Rolling Update

🟡 Interview standard · *maxSurge and maxUnavailable*

A Deployment with `replicas` pods updates in waves. Each wave may run up to `replicas + max_surge` pods and must keep at least `replicas − max_unavailable` available, so each wave replaces `max_surge + max_unavailable` pods. Return `(most pods at once, fewest available, waves)` for each setting.

**Examples**

- `replicas = 10, settings = [(1, 0), (2, 1), (0, 2), (25, 0)]` → `[(11, 10, 10), (12, 9, 4), (10, 8, 5), (20, 10, 1)]`

In [ ]:
def rolling_update(replicas, settings):
    """
    A Deployment with replicas pods updates in waves. Each wave may run up to
    replicas + max_surge pods and must keep at least replicas − max_unavailable
    available, so each wave replaces max_surge + max_unavailable pods. Return
    (most pods at once, fewest available, waves) for each setting.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rolling_update, [
    ((10, [(1, 0), (2, 1), (0, 2), (25, 0)]), [(11, 10, 10), (12, 9, 4), (10, 8, 5), (20, 10, 1)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate the waves with a counter.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Ceiling division.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Simulate the waves with a counter.

```python
def rolling_update(replicas, settings):
    out = []
    for surge, unavailable in settings:
        left, waves = replicas, 0
        while left > 0:
            left -= surge + unavailable
            waves += 1
        out.append((replicas + min(surge, replicas), replicas - unavailable, waves))
    return out
```

**Cost:** O(r) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Ceiling division.

```python
def rolling_update(replicas, settings):
    return [(replicas + min(s, replicas), replicas - u, math.ceil(replicas / (s + u))) for s, u in settings]
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate the waves with a counter. That costs O(r) time and O(1) space."

"I can do better: Ceiling division. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/plan-a-rolling-update)

<a id="q19"></a>
### Q19. Liveness vs Readiness Probes

🟡 Interview standard · *Restart or stop sending traffic?*

Every 10 s a pod's liveness and readiness probes run. After `threshold` consecutive **readiness** failures the pod is removed from the Service (no traffic) until readiness passes again; after `threshold` consecutive **liveness** failures the container is restarted (counters reset, and it's not ready until readiness passes). Return the list of `(second, action)` events: `'remove from service'`, `'add to service'`, `'restart'`.

**Examples**

- `results = [(True, True), (True, False), (True, False), (True, False), (True, True), (False, False), (False, False), (False, False), (True, True)], threshold = 3` → `[(40, 'remove from service'), (50, 'add to service'), (80, 'restart'), (90, 'add to service')]`

In [ ]:
def probes(results, threshold):
    """
    Every 10 s a pod's liveness and readiness probes run. After threshold
    consecutive readiness failures the pod is removed from the Service (no
    traffic) until readiness passes again; after threshold consecutive liveness
    failures the container is restarted (counters reset, and it's not ready
    until readiness passes). Return the list of (second, action) events: 'remove
    from service', 'add to service', 'restart'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(probes, [
    (([(True, True), (True, False), (True, False), (True, False), (True, True), (False, False), (False, False), (False, False), (True, True)], 3), [(40, 'remove from service'), (50, 'add to service'), (80, 'restart'), (90, 'add to service')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Counters and flags in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same state machine, compact.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Counters and flags in a loop.

```python
def probes(results, threshold):
    events, ready, live_fails, ready_fails = [], True, 0, 0
    for i, (live, rdy) in enumerate(results, 1):
        t = i * 10
        live_fails = 0 if live else live_fails + 1
        ready_fails = 0 if rdy else ready_fails + 1
        if live_fails >= threshold:
            events.append((t, "restart"))
            live_fails = ready_fails = 0
            if ready:
                ready = False
            continue
        if ready and ready_fails >= threshold:
            ready = False
            events.append((t, "remove from service"))
        elif not ready and rdy:
            ready = True
            events.append((t, "add to service"))
    return events
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same state machine, compact.

```python
def probes(results, threshold):
    ev, ready, lf, rf = [], True, 0, 0
    for t, (live, rdy) in zip(range(10, 10 * len(results) + 1, 10), results):
        lf, rf = (0 if live else lf + 1), (0 if rdy else rf + 1)
        if lf >= threshold:
            ev.append((t, "restart"))
            lf = rf = 0
            ready = False
        elif ready and rf >= threshold:
            ready = False
            ev.append((t, "remove from service"))
        elif not ready and rdy:
            ready = True
            ev.append((t, "add to service"))
    return ev
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Counters and flags in a loop. That costs O(n) time and O(1) space."

"I can do better: Same state machine, compact. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/liveness-vs-readiness-probes)

<a id="q20"></a>
### Q20. Share CPUs Fairly

🔴 Stretch · *Limits and contention*

Containers want `demand` CPUs but are capped at `limit`. The host has `cpus` cores. Allocate by **water-filling**: give every unsatisfied container an equal share, cap those that need less, and redistribute the leftover until all are satisfied or the CPUs are used up. Return allocations rounded to 2.

**Examples**

- `demand = {'api': 2.0, 'worker': 6.0, 'cron': 0.5, 'ml': 8.0}, limit = {'api': 2.0, 'worker': 4.0, 'cron': 1.0, 'ml': 8.0}, cpus = 8` → `{'api': 2.0, 'worker': 2.75, 'cron': 0.5, 'ml': 2.75}`
- `demand = {'a': 1.0, 'b': 1.0}, limit = {'a': 1.0, 'b': 0.5}, cpus = 4` → `{'a': 1.0, 'b': 0.5}`

In [ ]:
def cpu_share(demand, limit, cpus):
    """
    Containers want demand CPUs but are capped at limit. The host has cpus
    cores. Allocate by water-filling: give every unsatisfied container an equal
    share, cap those that need less, and redistribute the leftover until all are
    satisfied or the CPUs are used up. Return allocations rounded to 2.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cpu_share, [
    (({'api': 2.0, 'worker': 6.0, 'cron': 0.5, 'ml': 8.0}, {'api': 2.0, 'worker': 4.0, 'cron': 1.0, 'ml': 8.0}, 8), {'api': 2.0, 'worker': 2.75, 'cron': 0.5, 'ml': 2.75}),
    (({'a': 1.0, 'b': 1.0}, {'a': 1.0, 'b': 0.5}, 4), {'a': 1.0, 'b': 0.5}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeated equal splits until stable.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort by need and fill from the smallest.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Repeated equal splits until stable.

```python
def cpu_share(demand, limit, cpus):
    need = {c: min(demand[c], limit[c]) for c in demand}
    alloc = {c: 0.0 for c in demand}
    left = float(cpus)
    active = [c for c in need if need[c] > 0]
    while active and left > 1e-9:
        share = left / len(active)
        nxt = []
        for c in active:
            give = min(share, need[c] - alloc[c])
            alloc[c] += give
            left -= give
            if need[c] - alloc[c] > 1e-9:
                nxt.append(c)
        if len(nxt) == len(active):
            break
        active = nxt
    return {c: round(v, 2) for c, v in alloc.items()}
```

**Cost:** O(c²) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Sort by need and fill from the smallest.

```python
def cpu_share(demand, limit, cpus):
    need = sorted((min(demand[c], limit[c]), c) for c in demand)
    alloc, left = {}, float(cpus)
    for i, (n, c) in enumerate(need):
        give = min(n, left / (len(need) - i))
        alloc[c], left = give, left - give
    return {c: round(alloc[c], 2) for c in demand}
```

**Cost:** O(c log c) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeated equal splits until stable. That costs O(c²) time and O(c) space."

"I can do better: Sort by need and fill from the smallest. That's O(c log c) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/docker/share-cpus-fairly)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Why is my image so big?</b></summary>

A full base image, build tools left in, package caches, copying .git or data into the context, or many layers of deleted files. Use slim bases, multi-stage builds, .dockerignore and --no-cache-dir.

</details>

<details>
<summary><b>2. Why does a tiny code change rebuild everything?</b></summary>

An early COPY of the whole project invalidates every later layer. Copy and install dependencies first, then copy source code.

</details>

<details>
<summary><b>3. How do you keep data when a container is removed?</b></summary>

Store it in a named volume or a bind mount; the container's writable layer is deleted with the container.

</details>

<details>
<summary><b>4. How should secrets reach a container?</b></summary>

At runtime via environment variables from a secret store, mounted secret files, or BuildKit secrets during builds — never baked into an image layer.

</details>

<details>
<summary><b>5. Liveness vs readiness probe?</b></summary>

Liveness failing means the container is broken and gets restarted; readiness failing means it's alive but shouldn't receive traffic yet (warming up, dependency down).

</details>

<details>
<summary><b>6. Docker Compose or Kubernetes?</b></summary>

Compose for local development and small single-host deployments; Kubernetes when you need many machines, self-healing, autoscaling and rolling deploys.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Images, containers and your first Dockerfile
- [ ] I can explain: Fast, small and safe images
- [ ] I can explain: Running a stack with Docker Compose
- [ ] I can explain: From Docker to Kubernetes

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) When Is a Container Unhealthy?
- [ ] [Q17](#q17) Where Does the Data Go?
- [ ] [Q18](#q18) Plan a Rolling Update
- [ ] [Q19](#q19) Liveness vs Readiness Probes
- [ ] [Q20](#q20) Share CPUs Fairly

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.